# OOP Week 9 -- Factory & Registry Pattern

**Course:** Object-Oriented Programming (Year 2)
**Session:** 3 hours
**Prerequisites:** Weeks 1-8
**Focus:** creating objects from config, plugin architecture

---

## Learning Objectives

1. Explain the Factory Pattern and when to use it
2. Build a Registry that maps names to classes
3. Create objects from configuration dictionaries
4. Understand how this enables plugin architecture
5. Register new components without modifying existing code

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Section 1: What is a Factory?

A **Factory** is an object (or function) that **creates other objects**. Instead of using `ClassName(...)` directly, you ask the factory to build it for you.

### The Pizza Analogy

Instead of making pizza yourself, you tell the pizza shop: 'I want a Margherita.' The shop (factory) knows how to make it. Tomorrow they can add a new pizza type without changing how you order.

### In Our Pipeline

We want to create analyzers from a config file:
```json
{"analyzers": ["mean", "std", {"name": "events", "threshold": 50}]}
```
The factory reads this config and creates the right objects.

### The AnalyzerBase (from previous weeks)

In [ ]:
class AnalyzerBase:
    """Base class for analyzers."""
    def analyze(self, values):
        raise NotImplementedError

class MeanAnalyzer(AnalyzerBase):
    def analyze(self, values):
        return {"mean": round(sum(values)/len(values), 4)} if values else {}

class StdAnalyzer(AnalyzerBase):
    def analyze(self, values):
        if not values:
            return {}
        m = sum(values) / len(values)
        return {"std": round((sum((x-m)**2 for x in values)/len(values))**0.5, 4)}

class EventAnalyzer(AnalyzerBase):
    def __init__(self, threshold=50):
        self.threshold = threshold
    def analyze(self, values):
        return {"events_above": sum(1 for v in values if v > self.threshold)}

print("Analyzer classes defined.")

**Expected Output:**
```
Analyzer classes defined.
```

---
## Section 2: The Registry Pattern

In [ ]:
class AnalyzerFactory:
    """Factory + Registry for creating analyzers from config."""

    _registry = {}

    @classmethod
    def register(cls, name, analyzer_class):
        """Register an analyzer class under a name."""
        cls._registry[name] = analyzer_class
        print("Registered: " + name + " -> " + analyzer_class.__name__)

    @classmethod
    def create(cls, name, **kwargs):
        """Create an analyzer by name."""
        if name not in cls._registry:
            available = list(cls._registry.keys())
            raise ValueError("Unknown analyzer: " + name
                           + ". Available: " + str(available))
        return cls._registry[name](**kwargs)

    @classmethod
    def list_available(cls):
        """List all registered analyzers."""
        return list(cls._registry.keys())


# Register built-in analyzers
AnalyzerFactory.register("mean", MeanAnalyzer)
AnalyzerFactory.register("std", StdAnalyzer)
AnalyzerFactory.register("events", EventAnalyzer)

print("Available:", AnalyzerFactory.list_available())

**Expected Output:**
```
Registered: mean -> MeanAnalyzer
Registered: std -> StdAnalyzer
Registered: events -> EventAnalyzer
Available: ['mean', 'std', 'events']
```

---
## Section 3: Creating Objects from Config

In [ ]:
# Config (could be loaded from JSON/YAML)
config = {
    "analyzers": [
        {"name": "mean"},
        {"name": "std"},
        {"name": "events", "params": {"threshold": 40}},
    ]
}

# Build analyzers from config
analyzers = []
for spec in config["analyzers"]:
    params = spec.get("params", {})
    a = AnalyzerFactory.create(spec["name"], **params)
    analyzers.append(a)
    print("Created: " + spec["name"] + " -> " + type(a).__name__)

# Run all
values = [10, 25, 30, 55, 20, 45, 60]
combined = {}
for a in analyzers:
    combined.update(a.analyze(values))

print()
print("Results:", combined)

**Expected Output:**
```
Created: mean -> MeanAnalyzer
Created: std -> StdAnalyzer
Created: events -> EventAnalyzer

Results: {'mean': 35.0, 'std': 16.5831, 'events_above': 3}
```

---
## Section 4: Plugin Architecture

The factory + registry enables a **plugin** architecture. Anyone can add new analyzers without modifying existing code -- just create a new class and register it.

In [ ]:
# A plugin: new analyzer, no changes to existing code!
class PercentileAnalyzer(AnalyzerBase):
    def __init__(self, percentiles=None):
        self.percentiles = percentiles or [25, 50, 75]
    def analyze(self, values):
        if not values:
            return {}
        s = sorted(values)
        n = len(s)
        result = {}
        for p in self.percentiles:
            idx = min(int(n * p / 100), n - 1)
            result["p" + str(p)] = s[idx]
        return result

# One line to register!
AnalyzerFactory.register("percentile", PercentileAnalyzer)

# Now it works with the existing pipeline
p = AnalyzerFactory.create("percentile", percentiles=[10, 50, 90])
print("Percentile results:", p.analyze(values))
print()
print("Available analyzers:", AnalyzerFactory.list_available())

**Expected Output:**
```
Registered: percentile -> PercentileAnalyzer
Percentile results: {'p10': 10, 'p50': 30, 'p90': 55}

Available analyzers: ['mean', 'std', 'events', 'percentile']
```

---
### Try It!

Create a `VarianceAnalyzer(AnalyzerBase)`, register it with the factory, then create it from config and run it.

In [ ]:
# YOUR CODE HERE


---
## Mini-Quiz

In [ ]:
# Q1: What is a Factory?
# Answer: 

# Q2: What is a Registry?
# Answer: 

# Q3: How does the factory enable plugin architecture?
# Answer: 

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)